# Getting Started with FlowBook

**FlowBook** is a JupyterLab extension that turns your notebook into a
*reproducible* notebook.

This tutorial is self-contained: go through it top to bottom, and it
will walk you through the key ideas and show you live how FlowBook
reacts to different editing patterns.

## What problem does FlowBook solve?

Jupyter notebooks let you run cells in any order. That flexibility is
what makes them great for exploration, but it also makes it easy to
end up in a state where the *visible* outputs no longer match what you
would get if you re-ran the notebook from scratch. Cells accumulate
hidden state, implicit dependencies form, and a notebook that *looks*
consistent can silently be reporting stale or contradictory results.

FlowBook tracks what each cell **reads** and **writes**, watches how
edits and re-executions affect those dependencies, and warns you (or
stops you) whenever something you've done breaks the correspondence
between the notebook's recorded outputs and a fresh top-to-bottom
execution.

## Key vocabulary

- **Reproducible notebook**: executing its cells sequentially from an
  empty store produces exactly the outputs currently recorded.
- **Re-run consistent cell**: re-executing this cell from the current
  store would produce the same output it produced last time. A cell is
  **clean** if it has been executed and is re-run consistent.
- **Stale cell**: the cell's inputs may have changed since it last ran,
  so its recorded output may no longer reflect the current state.
  FlowBook marks stale cells visually so you know what to re-run.
- **Validity predicates**: conditions on the read/write sets that must
  hold for a cell to become clean. If they fail, FlowBook rejects the
  execution and rolls back the namespace.

When *every* cell is clean, the notebook is reproducible.

## Cell labels

Throughout this tutorial we refer to code cells by letter: @A, @B,
@C, and so on, in the order they appear in the notebook. FlowBook
displays the same letter in the cell gutter, so you can always tell
which cell is which.

## How this tutorial is organized

Each part has a short explanation, a small example, and instructions
telling you what to edit or run. Work through them in order; some
later parts build on state from earlier ones.

1. Basic read/write tracking (@A to @C)
2. Forward staleness (editing upstream)
3. Edit detection without running (@D to @E)
4. Running cells out of order (@F to @G)
5. Rejected executions and rollback (@H to @J)
6. Column-level tracking with pandas (@K to @Q)
7. Unrecoverable mutations (@R to @T)

## Part 1: Basic read/write tracking

FlowBook records the set of variables each cell **reads** (R) and
**writes** (W). After you run a cell, its status line shows these sets
along with timing information.

Run cells **@A**, **@B**, and **@C** below in order. Look at the status
line beneath each: you should see that @A writes `price`, @B reads
`price` and writes `tax`, and @C reads both and writes `total`.

### @A: define `price`

In [ ]:
price = 100

### @B: compute `tax` from `price`

In [ ]:
tax = price * 0.08
print(f"Tax: ${tax:.2f}")

### @C: compute `total` from `price` and `tax`

In [ ]:
total = price + tax
print(f"Total: ${total:.2f}")

After running all three, every cell is **clean**. The notebook is
reproducible: a fresh top-to-bottom run would produce exactly these
outputs.

## Part 2: Forward staleness

Now edit **@A** above so that `price = 200`, and re-run *only* @A.

**Expected:** @B and @C turn yellow. They are now **stale**: they read
`price`, and `price` has been rewritten, so their displayed outputs no
longer match what re-running them would produce.

This is called **forward staleness**: re-executing cell *i* can mark
any cell *j > i* stale if *j* reads or writes something *i* wrote.
Staleness also propagates transitively: @C becomes stale because it
reads `tax`, which is written by (now-stale) @B.

Re-run @B and @C to make them clean again, then continue.

## Part 3: Edit detection without running

You don't have to *run* a cell for FlowBook to notice a change.
Editing a previously-executed cell is enough: after a short debounce
(about one second), FlowBook marks the cell, and anything downstream
of it, as stale.

Run **@D** and **@E** below in order. Then go back to @D and change
`rate` to `0.07`, but **don't run it**. Wait a second.

**Expected:** @D turns stale (its source no longer matches what was
executed) and @E also becomes stale (it depends on @D). Re-run both
to return to a clean state.

### @D: a rate

In [ ]:
rate = 0.05

### @E: use the rate

In [ ]:
interest = 1000 * rate
print(f"Interest: ${interest:.2f}")

## Part 4: Running cells out of order

Jupyter happily lets you run cells in any order. FlowBook tolerates
this during exploration but marks the result as stale: the notebook is
not yet reproducible because a top-to-bottom replay would behave
differently.

Try this: run **@G first**, then **@F**.

**Expected:** @F executes, but it's marked stale. It read `msg` from
@G, and @G sits below @F in the notebook. A fresh top-to-bottom run
would fail because `msg` wouldn't exist yet when @F ran. FlowBook
flags this as a re-run consistency problem.

### @F: uses `msg`

In [ ]:
print(msg.upper())

### @G: defines `msg`

In [ ]:
msg = "hello flowbook"

To recover, either move @G above @F (use JupyterLab's cell drag
handle), or delete @F and rewrite it below @G.

## Part 5: Rejected executions and rollback

Some edits don't just create staleness; they create an *irrecoverable*
inconsistency. FlowBook enforces four **validity predicates** every
time a cell runs:

1. A cell does not read its own writes.
2. Every read was written by some earlier cell.
3. A cell does not read variables written by later cells.
4. A cell does not overwrite variables read by earlier cells.

If any predicate fails, FlowBook **rejects the execution and rolls
back the namespace**. The cell's side effects are undone so you don't
end up with half-applied state.

Run cells **@H**, **@I**, **@J** in order.

**Expected:** @H and @I succeed. @J is **rejected** because it tries
to overwrite `df`, which @I already read. Allowing it would make @I's
output inconsistent with any top-to-bottom execution. The rejection
includes a rollback: `df` still holds the original DataFrame.

### @H: load data

In [ ]:
import pandas as pd
df = pd.DataFrame({'x': [1, 2, 3], 'y': [4, 5, 6]})

### @I: read from `df`

In [ ]:
total = df['x'].sum()
print(f"Sum of x: {total}")

### @J: rebinds `df` (will be rejected)

In [ ]:
df = pd.DataFrame({'x': [100, 200, 300]})

The fix is to use a new name (e.g. `df2 = pd.DataFrame(...)`) or move
the new data definition *above* @I.

## Part 6: Column-level tracking with pandas

A DataFrame is a common unit of state in data science, and a lot of
reproducibility mistakes arise from editing *one* column while other
cells read *other* columns. FlowBook tracks pandas DataFrames at the
**column** level, not just the DataFrame level. Two cells that touch
different columns of the same DataFrame do not conflict.

### Scenario A: independent columns (no conflict)

Run **@K**, **@L**, **@M** in order.

**Expected:** all three succeed. @L only reads `age`; @M writes a new
column `adjusted_age`. No overlap, so no conflict.

### @K: build a DataFrame

In [ ]:
import pandas as pd
people = pd.DataFrame({
    'name': ['Alice', 'Bob'],
    'age': [30, 25],
    'score': [85, 92],
})

### @L: read the `age` column

In [ ]:
avg_age = people['age'].mean()
print(f"Average age: {avg_age}")

### @M: add a new column

In [ ]:
people['adjusted_age'] = people['age'] * 1.1  # different column, OK

### Scenario B: same column (conflict)

Now run **@N**, **@O**, **@P** in order.

**Expected:** @P is flagged. It would overwrite `age`, which @O read.
That's a violation of predicate 4 (no write after read), exactly like
the `df` rebinding in Part 5, just at column granularity.

### @N: a smaller DataFrame

In [ ]:
import pandas as pd
employees = pd.DataFrame({
    'name': ['Alice', 'Bob'],
    'age': [30, 25],
})

### @O: read `age`

In [ ]:
avg_age_emp = employees['age'].mean()
print(f"Average age: {avg_age_emp}")

### @P: overwrite `age` (will be rejected)

In [ ]:
employees['age'] = employees['age'] + 1

Typical fix: make the change on a copy so the original DataFrame
remains unchanged, or move the transformation to happen *before* any
cell that reads `age`.

### @Q: corrected version (on a copy)

In [ ]:
from copy import deepcopy
employees_adj = deepcopy(employees)
employees_adj['age'] = employees_adj['age'] + 1
print(employees_adj)

## Part 7: Unrecoverable mutations (in-place vs. rebinding)

FlowBook restores reproducibility by **re-running cells**. For that to
work, a cell's effect on the store has to be something a re-run can
reproduce from scratch.

- A **rebinding** like `x = [1, 2, 3]` is always *recoverable*:
  re-running the cell reconstructs `x` from scratch.
- An **in-place mutation** like `x[0] = 99` is *unrecoverable*:
  FlowBook can't restore the previous value by re-running the cell,
  because re-running just redoes the mutation on whatever is currently
  there.

FlowBook warns when a cell performs an unrecoverable mutation without
rebinding the variable.

Run **@R** and **@S** in order.

### @R: build a list

In [ ]:
nums = [1, 2, 3, 4, 5]

### @S: in-place mutation (warning)

In [ ]:
nums[0] = 99  # in-place mutation; FlowBook flags this
print(nums)

The recoverable version does the same thing by rebinding: the result
is a *new* list object, which FlowBook can safely reproduce on
re-run.

### @T: rebinding version

In [ ]:
nums = [99, 2, 3, 4, 5]
print(nums)

The same idea applies to NumPy arrays and DataFrames:

| Operation | Recoverable? |
|---|---|
| `x = [1, 2, 3]` | Yes, rebinding |
| `x[0] = 99` (list/array) | No, in-place mutation |
| `df['col'] = ...` | Yes, column-level tracking |
| `df.values[i, j] = ...` | No, bypasses column tracking |
| `df.reset_index(inplace=True)` | No, structural mutation |

Best practice: prefer rebindings, or contain in-place mutations to a
single cell that *also* rebinds the variable (e.g. load, mutate, and
rebind all together).

## Summary

You've now seen the core of FlowBook:

- Every cell has tracked **read** and **write** sets, shown in the
  status line after execution.
- Edits and re-executions propagate **forward** and **backward**
  staleness so you always know which cells to re-run.
- FlowBook rejects executions that would break re-run consistency,
  rolling back the namespace so your state stays coherent.
- DataFrames are tracked at the **column** level, so unrelated column
  work doesn't create false conflicts.
- **Rebindings are recoverable; in-place mutations are not.** Prefer
  rebindings where you can.

When every cell is clean, your notebook is reproducible: running it
top-to-bottom from an empty kernel will produce exactly the outputs
you see now.